# SFT

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display, HTML, Markdown
import torch
from trl import SFTConfig, SFTTrainer
import math

model_id = "LiquidAI/LFM2.5-350M" # <- or LFM2-700M or LFM2-350M
normalized_model_id = model_id.replace("/", "_").replace("-", "_").replace(".", "_")

/home/ivan.gonzalez/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
torch.cuda.is_available()

True

In [3]:
print("📚 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

📚 Loading tokenizer...


In [4]:
print("🧠 Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype="bfloat16",
    # attn_implementation="flash_attention_2" # <- uncomment on compatible GPU
    attn_implementation="sdpa",
)

print("✅ Local model loaded successfully!")
print(f"🔢 Parameters: {model.num_parameters():,}")
print(f"📖 Vocab size: {len(tokenizer)}")
print(f"💾 Model size: ~{model.num_parameters() * 2 / 1e9:.1f} GB (bfloat16)")

🧠 Loading model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1092.17it/s]


✅ Local model loaded successfully!
🔢 Parameters: 354,483,968
📖 Vocab size: 64402
💾 Model size: ~0.7 GB (bfloat16)


In [5]:
from datasets import load_dataset, Dataset

print("📥 Loading SFT dataset...")
dataset = load_dataset("thinkPy/paraguay-cultural-alignment", "sft")

📥 Loading SFT dataset...


Generating validation split: 100%|██████████| 300/300 [00:00<00:00, 38747.65 examples/s]


In [6]:
dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'response', 'topic', 'cultural', 'score'],
        num_rows: 2700
    })
    validation: Dataset({
        features: ['prompt', 'response', 'topic', 'cultural', 'score'],
        num_rows: 300
    })
})

In [7]:
def format(example):
    return {
        "messages": [
            {"role": "user", "content": example['prompt']},
            {"role": "assistant", "content": example['response']}
        ]
    }

dataset_sft = dataset.map(format, remove_columns=dataset["train"].column_names)

Map: 100%|██████████| 300/300 [00:00<00:00, 9748.38 examples/s]


In [8]:
dataset_sft

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 2700
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 300
    })
})

In [9]:
train_dataset_sft = dataset_sft["train"]
val_dataset_sft = dataset_sft["validation"]

print("✅ SFT Dataset loaded:")
print(f"   📚 Train samples: {len(train_dataset_sft)}")
print(f"   🧪 Eval samples: {len(val_dataset_sft)}")
print(f"\n📝 Single Sample: {train_dataset_sft[0]['messages']}")

✅ SFT Dataset loaded:
   📚 Train samples: 2700
   🧪 Eval samples: 300

📝 Single Sample: [{'role': 'user', 'content': 'Bailar bajo la lluvia, la melodía comienza,\nUna sinfonía de precipitación, música en nuestros corazones,\nEl golpeteo de las gotas de lluvia, llena el aire,\nEl sonido de la naturaleza, una orquesta tan rara.\n\nLas gotas de lluvia golpean el pavimento y cantan,\nUna dulce percusión, el sonido que trae,\nEl ritmo de la lluvia al caer sobre los techos,\nEs como un ritmo de tambor, un sonido que ladra.\n\nGolpea contra las hojas de los árboles,\nUna melodía como el susurro de las llaves,\nComo si la lluvia estuviera tocando una melodía,\nUna armonía que nos deja en un estado de éxtasis.\n\nEl sonido de la lluvia en diferentes superficies,\nEs como una danza de música, emerge,\nUna sinfonía acústica, la lluvia y nosotros,\nUn medley melodioso, un coro divino.\n\nY mientras bailamos, la música a nuestro alrededor se eleva,\nUna celebración de la vida, un momento que sobres

In [10]:
non_reasoning_dataset = train_dataset_sft.map(lambda x: {"text": tokenizer.apply_chat_template(
    x["messages"],
    tokenize = False,
)})

Map: 100%|██████████| 2700/2700 [00:00<00:00, 4524.47 examples/s]


In [11]:
non_reasoning_dataset[0]

{'messages': [{'role': 'user',
   'content': 'Bailar bajo la lluvia, la melodía comienza,\nUna sinfonía de precipitación, música en nuestros corazones,\nEl golpeteo de las gotas de lluvia, llena el aire,\nEl sonido de la naturaleza, una orquesta tan rara.\n\nLas gotas de lluvia golpean el pavimento y cantan,\nUna dulce percusión, el sonido que trae,\nEl ritmo de la lluvia al caer sobre los techos,\nEs como un ritmo de tambor, un sonido que ladra.\n\nGolpea contra las hojas de los árboles,\nUna melodía como el susurro de las llaves,\nComo si la lluvia estuviera tocando una melodía,\nUna armonía que nos deja en un estado de éxtasis.\n\nEl sonido de la lluvia en diferentes superficies,\nEs como una danza de música, emerge,\nUna sinfonía acústica, la lluvia y nosotros,\nUn medley melodioso, un coro divino.\n\nY mientras bailamos, la música a nuestro alrededor se eleva,\nUna celebración de la vida, un momento que sobresale,\nEl sonido de las gotas de lluvia, una melodía dulce,\nUn baile baj

In [12]:
num_train_epochs = 3
per_device_train_batch_size = 2
per_device_eval_batch_size = 1
gradient_accumulation_steps = 4
num_devices=1

num_train_examples = len(train_dataset_sft)
num_eval_examples = len(val_dataset_sft)

# Batch size efectivo de entrenamiento
effective_train_batch_size = (
    per_device_train_batch_size
    * gradient_accumulation_steps
    * num_devices
)

# Batch size efectivo de evaluación
effective_eval_batch_size = (
    per_device_eval_batch_size
    * num_devices
)

# Steps de entrenamiento
steps_per_epoch = math.ceil(
    num_train_examples / effective_train_batch_size
)

total_training_steps = steps_per_epoch * num_train_epochs

# Steps de evaluación: cuántos batches tiene validation
eval_steps_per_run = math.ceil(
    num_eval_examples / effective_eval_batch_size
)

print("Train examples:", num_train_examples)
print("Validation examples:", num_eval_examples)
print()
print("Train effective batch size:", effective_train_batch_size)
print("Eval effective batch size:", effective_eval_batch_size)
print()
print("Steps por epoch:", steps_per_epoch)
print("Total training steps:", total_training_steps)
print("Eval batches por evaluación:", eval_steps_per_run)

Train examples: 2700
Validation examples: 300

Train effective batch size: 8
Eval effective batch size: 1

Steps por epoch: 338
Total training steps: 1014
Eval batches por evaluación: 300


In [13]:
import os

tb_log_dir = f"./train/sft/{normalized_model_id}/logs"
os.environ["TENSORBOARD_LOGGING_DIR"] = tb_log_dir

sft_config = SFTConfig(
    output_dir=f"./train/{normalized_model_id}",

    num_train_epochs=num_train_epochs,

    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,

    learning_rate=5e-5,
    lr_scheduler_type="cosine",

    warmup_steps=math.ceil(total_training_steps * 0.01),

    logging_strategy="steps",
    logging_steps=math.ceil(total_training_steps * 0.05),

    save_strategy="steps",
    save_steps=math.ceil(total_training_steps * 0.05),#save_steps,
    save_total_limit=3,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    eval_strategy="steps",
    eval_steps=math.ceil(total_training_steps * 0.05),

    report_to=["tensorboard"],

    dataset_text_field="messages",
    bf16=True,
)

/tmp/ipykernel_1274234/1472728366.py:6: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  sft_config = SFTConfig(


In [15]:
pip install tensorboard --break-system-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 3.6 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 3.6 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 3.6 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 3.6 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [tensorboard] [tensorboard]data-server]

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: /home/ivan.gonzalez/venv/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [16]:
print("🏗️  Creating SFT trainer...")
sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset_sft,
    eval_dataset=val_dataset_sft,
    processing_class=tokenizer,
)

print("\n🚀 Starting SFT training...")
sft_trainer.train()

print("🎉 SFT training completed!")

🏗️  Creating SFT trainer...

🚀 Starting SFT training...


Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
51,2.152187,1.810662,1.793120,0.604568,188783.000000
102,1.635806,1.569515,1.513843,0.652156,375384.000000
153,1.416430,1.433478,1.302866,0.678546,566056.000000
204,1.323617,1.293732,1.301968,0.708136,755603.000000
255,1.147584,1.160353,1.066900,0.738553,940878.000000
306,1.053133,1.066598,1.054659,0.758832,1125089.000000
357,0.828067,1.019225,0.854110,0.773565,1313263.000000
408,0.547010,0.964031,0.802873,0.788023,1497742.000000
459,0.559470,0.918756,0.795346,0.800196,1686349.000000
510,0.507000,0.894161,0.723913,0.808303,1875754.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.68it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


🎉 SFT training completed!


In [18]:
pip install python-dotenv --break-system-packages


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: /home/ivan.gonzalez/venv/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [20]:
pip install ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 914.9/914.9 kB 3.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 3.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]3 [ipywidgets]

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: /home/ivan.gonzalez/venv/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [33]:
import os
from dotenv import load_dotenv
from huggingface_hub import login, HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

login(token=HF_TOKEN)

#https://huggingface.co/LiquidAI/LFM2.5-350M
# Repo destino
hub_model_id = "thinkPy/sft_LFM2_5_350M"

# Usa tu mejor checkpoint DPO
best_checkpoint_path = "./train/LiquidAI_LFM2_5_350M/checkpoint-1014"
# Si tu checkpoint real tiene otro número, cámbialo aquí.

# Cargar modelo y tokenizer desde el checkpoint
model = AutoModelForCausalLM.from_pretrained(
    best_checkpoint_path,
    device_map="auto",
    dtype="bfloat16",
)

'''best_checkpoint_path = os.path.abspath("./train/sft/LiquidAI_LFM2_5_350M/checkpoint-1014/")

model = AutoModelForCausalLM.from_pretrained(
    best_checkpoint_path,
    device_map="auto",
    dtype="bfloat16",
)'''

'''best_checkpoint_path = os.path.abspath("./train/sft/LiquidAI_LFM2_5_350M/checkpoint-1014")

config = AutoConfig.from_pretrained(best_checkpoint_path)

model = AutoModelForCausalLM.from_pretrained(
    best_checkpoint_path,
    config=config,
    device_map="auto",
    torch_dtype="bfloat16",   # <-- nota: es torch_dtype, no dtype
    local_files_only=True,    # fuerza a buscar solo localmente
)'''

#tokenizer = AutoTokenizer.from_pretrained(best_checkpoint_path, local_files_only=True)

tokenizer = AutoTokenizer.from_pretrained(best_checkpoint_path)


# Recomendado para inferencia
model.config.use_cache = True

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 789.55it/s]


In [34]:
import os

model.push_to_hub(
    hub_model_id,
    token=HF_TOKEN,
    private=False,
)

tokenizer.push_to_hub(
    hub_model_id,
    token=HF_TOKEN,
    private=False,
)

print(f"✅ Modelo subido a: https://huggingface.co/{hub_model_id}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.53it/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):   0%|          |  611kB /  709MB, 57.3kB/s  
Processing Files (0 / 1):   0%|          | 1.85MB /  709MB,  170kB/s  
Processing Files (0 / 1):   0%|          | 3.07MB /  709MB,  280kB/s  
Processing Files (0 / 1):   1%|          | 3.69MB /  709MB,  329kB/s  
Processing Files (0 / 1):   1%|          | 4.94MB /  709MB,  440kB/s  
Processing Files (0 / 1):   1%|          | 6.16MB /  709MB,  547kB/s  
Processing Files (0 / 1):   1%|          | 7.39MB /  709MB,  634kB/s  
Processing Files (0 / 1):   1%|▏         | 9.25MB /  709MB,  778kB/s  
Processing Files (0 / 1):   2%|▏         | 11.1MB /  709MB,  914kB/s  
Processing Files (0 / 1):   2%|▏         | 11.7MB /  709MB,  935kB/s  
Processing Files (0 / 1):   2%|▏         | 13.0MB /  709MB, 1.04MB/s  
Processing Files (0 / 1):   2%|▏         | 14.8MB /  709MB, 1.16MB/s  
Processing Fi

✅ Modelo subido a: https://huggingface.co/thinkPy/sft_LFM2_5_350M


# DPO

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display, HTML, Markdown
import torch
from trl import SFTConfig, SFTTrainer
import math

model_id = "LiquidAI/LFM2.5-350M"
normalized_model_id = model_id.replace("/", "_").replace("-", "_").replace(".", "_")
load_model_id = f"./train/{normalized_model_id}/checkpoint-1014"

In [2]:
print("📚 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(load_model_id)

📚 Loading tokenizer...


In [3]:
print("🧠 Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    load_model_id,
    device_map="auto",
    dtype="bfloat16",
    # attn_implementation="flash_attention_2" # <- uncomment on compatible GPU
    attn_implementation="sdpa",
)

print("✅ Local model loaded successfully!")
print(f"🔢 Parameters: {model.num_parameters():,}")
print(f"📖 Vocab size: {len(tokenizer)}")
print(f"💾 Model size: ~{model.num_parameters() * 2 / 1e9:.1f} GB (bfloat16)")

🧠 Loading model...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

✅ Local model loaded successfully!
🔢 Parameters: 354,483,968
📖 Vocab size: 64402
💾 Model size: ~0.7 GB (bfloat16)


In [4]:
from datasets import load_dataset, Dataset

print("📥 Loading SFT dataset...")
dataset = load_dataset("thinkPy/paraguay-cultural-alignment", "dpo")

📥 Loading SFT dataset...


dpo/train-00000-of-00001.parquet:   0%|          | 0.00/1.42M [00:00<?, ?B/s]

dpo/validation-00000-of-00001.parquet:   0%|          | 0.00/256k [00:00<?, ?B/s]

dpo/test-00000-of-00001.parquet:   0%|          | 0.00/376k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/300 [00:00<?, ? examples/s]

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'topic', 'cultural_chosen', 'cultural_rejected', 'score_chosen', 'score_rejected'],
        num_rows: 1500
    })
    validation: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'topic', 'cultural_chosen', 'cultural_rejected', 'score_chosen', 'score_rejected'],
        num_rows: 200
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'rejected', 'topic', 'cultural_chosen', 'cultural_rejected', 'score_chosen', 'score_rejected'],
        num_rows: 300
    })
})

In [6]:
def format_dpo_plain(example):
    prompt = example["prompt"].strip()
    chosen = example["chosen"].strip()
    rejected = example["rejected"].strip()

    return {
        "prompt": prompt + " \n",
        "chosen": chosen,
        "rejected": rejected,
    }

dataset_dpo = dataset.map(format_dpo_plain)
dataset_dpo = dataset_dpo.select_columns(["prompt", "chosen", "rejected"])

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [7]:
dataset_dpo

DatasetDict({
    train: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 1500
    })
    validation: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 200
    })
    test: Dataset({
        features: ['prompt', 'chosen', 'rejected'],
        num_rows: 300
    })
})

In [8]:
dataset_dpo["train"][0]

{'prompt': '¡Muchas gracias por sus amables palabras! Me alegra que disfrutara del poema. En cuanto a los hombres lobo, eran en efecto una presencia amenazante en el bosque encantado. Su pelaje era tan negro como el cielo nocturno, y sus ojos brillaban como bolas de fuego, advirtiendo al aventurero de su hambre y sed de sangre implacables.\n\nSu comportamiento era el de depredadores alfa, astutos y brutales en sus ataques. Acechaban a sus presas con una gracia feral, arrastrándose silenciosamente por la maleza hasta que estaban lo suficientemente cerca como para saltar. Una vez que atacaban, eran implacables, golpeando con toda la fuerza de sus garras y dientes afilados como navajas.\n\nEl aventurero fue sorprendido por su ferocidad, pero no fue fácilmente derrotado. Con su espada en la mano, luchó con todas sus fuerzas, esquivando y zigzagueando entre los árboles mientras los hombres lobo se lanzaban hacia él desde diferentes direcciones.\n\nSu batalla fue agotadora, ya que los hombre

In [9]:
train_dataset_dpo = dataset_dpo["train"]
val_dataset_dpo = dataset_dpo["validation"]
test_dataset_dpo = dataset_dpo["test"]

print("✅ DPO Dataset loaded:")
print(f"   📚 Train samples: {len(train_dataset_dpo)}")
print(f"   🧪 Eval samples: {len(val_dataset_dpo)}")
print(f"\n📝 Single Sample: {train_dataset_dpo[0]['prompt']}")

✅ DPO Dataset loaded:
   📚 Train samples: 1500
   🧪 Eval samples: 200

📝 Single Sample: ¡Muchas gracias por sus amables palabras! Me alegra que disfrutara del poema. En cuanto a los hombres lobo, eran en efecto una presencia amenazante en el bosque encantado. Su pelaje era tan negro como el cielo nocturno, y sus ojos brillaban como bolas de fuego, advirtiendo al aventurero de su hambre y sed de sangre implacables.

Su comportamiento era el de depredadores alfa, astutos y brutales en sus ataques. Acechaban a sus presas con una gracia feral, arrastrándose silenciosamente por la maleza hasta que estaban lo suficientemente cerca como para saltar. Una vez que atacaban, eran implacables, golpeando con toda la fuerza de sus garras y dientes afilados como navajas.

El aventurero fue sorprendido por su ferocidad, pero no fue fácilmente derrotado. Con su espada en la mano, luchó con todas sus fuerzas, esquivando y zigzagueando entre los árboles mientras los hombres lobo se lanzaban hacia él desd

In [10]:
num_train_epochs = 1
per_device_train_batch_size = 1
per_device_eval_batch_size = 1
gradient_accumulation_steps = 4
num_devices=1

num_train_examples = len(train_dataset_dpo)
num_eval_examples = len(val_dataset_dpo)

# Batch size efectivo de entrenamiento
effective_train_batch_size = (
    per_device_train_batch_size
    * gradient_accumulation_steps
    * num_devices
)

# Batch size efectivo de evaluación
effective_eval_batch_size = (
    per_device_eval_batch_size
    * num_devices
)

# Steps de entrenamiento
steps_per_epoch = math.ceil(
    num_train_examples / effective_train_batch_size
)

total_training_steps = steps_per_epoch * num_train_epochs

# Steps de evaluación: cuántos batches tiene validation
eval_steps_per_run = math.ceil(
    num_eval_examples / effective_eval_batch_size
)

warmup_steps=math.ceil(total_training_steps * 0.03)
logging_steps=math.ceil(total_training_steps * 0.01)
eval_steps=math.ceil(total_training_steps * 0.05)
save_steps=math.ceil(total_training_steps * 0.05)

print("Train examples:", num_train_examples)
print("Validation examples:", num_eval_examples)
print()
print("Train effective batch size:", effective_train_batch_size)
print("Eval effective batch size:", effective_eval_batch_size)
print()
print("Steps por epoch:", steps_per_epoch)
print("Total training steps:", total_training_steps)
print("Eval batches por evaluación:", eval_steps_per_run)
print()
print("Warmup Steps:", warmup_steps)
print("Logging Steps:", logging_steps)
print("Eval Steps:", eval_steps)
print("Save Steps:", save_steps)

Train examples: 1500
Validation examples: 200

Train effective batch size: 4
Eval effective batch size: 1

Steps por epoch: 375
Total training steps: 375
Eval batches por evaluación: 200

Warmup Steps: 12
Logging Steps: 4
Eval Steps: 19
Save Steps: 19


In [12]:
import os
from trl import DPOConfig, DPOTrainer

# model.config.use_cache = False

output_dir = f"./train/dpo/{normalized_model_id}"
os.environ["TENSORBOARD_LOGGING_DIR"] = f"{output_dir}/logs"

dpo_config = DPOConfig(
    output_dir=output_dir,

    num_train_epochs=num_train_epochs,

    per_device_train_batch_size=per_device_train_batch_size,
    per_device_eval_batch_size=per_device_eval_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,

    # DPO suele usar LR mucho más bajo que SFT
    learning_rate=1e-6,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,

    # DPO estándar
    beta=0.1,
    loss_type="sigmoid",

    # Logging / eval / checkpoints
    logging_strategy="steps",
    logging_steps=logging_steps,

    eval_strategy="steps",
    eval_steps=eval_steps,

    save_strategy="steps",
    save_steps=save_steps,
    save_total_limit=3,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # gradient_checkpointing=True,

    report_to=["tensorboard"],

    bf16=True,
)

In [13]:
print("🏗️ Creating DPO trainer...")

dpo_trainer = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=train_dataset_dpo,
    eval_dataset=val_dataset_dpo,
    processing_class=tokenizer,
)


🏗️ Creating DPO trainer...


Adding EOS to train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [14]:
print("\n🚀 Starting DPO training...")
train_result = dpo_trainer.train()

# model.config.use_cache = True

print("🎉 DPO training completed!")


🚀 Starting DPO training...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
19,0.665710,0.671545,0.849952,68055.000000,-1.972282,-2.000041,0.708665,-0.011142,-0.058096,0.665000,0.046954,-196.132588,-346.465420
38,0.604477,0.613017,0.848523,135093.000000,-1.971992,-1.999884,0.708900,-0.021191,-0.193093,0.915000,0.171902,-196.233082,-347.815389
57,0.587935,0.559245,0.847322,207042.000000,-1.971531,-1.999265,0.709300,-0.022334,-0.318197,0.970000,0.295864,-196.244504,-349.066430
76,0.495070,0.510549,0.846049,272372.000000,-1.971003,-1.998866,0.708872,-0.031713,-0.451790,0.990000,0.420077,-196.338296,-350.402359
95,0.499426,0.480228,0.845179,338901.000000,-1.970984,-1.998727,0.709018,-0.052644,-0.555135,0.995000,0.502490,-196.547613,-351.435805
114,0.430570,0.452203,0.844213,408759.000000,-1.970095,-1.998232,0.709292,-0.063671,-0.646261,0.995000,0.582590,-196.657879,-352.347069
133,0.419134,0.429165,0.843972,471605.000000,-1.970194,-1.998130,0.709304,-0.065575,-0.717623,0.995000,0.652048,-196.676922,-353.060689
152,0.437016,0.410190,0.843218,536736.000000,-1.970031,-1.997812,0.708991,-0.065767,-0.777763,1.000000,0.711996,-196.678837,-353.662083
171,0.423691,0.395144,0.842703,606261.000000,-1.969747,-1.997630,0.709173,-0.073100,-0.833900,0.995000,0.760800,-196.752170,-354.223458
190,0.382993,0.388365,0.842039,676170.000000,-1.969360,-1.997168,0.709274,-0.079211,-0.864479,0.995000,0.785268,-196.813281,-354.529251


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


🎉 DPO training completed!


In [15]:
test_dataset_dpo

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 300
})

In [16]:
from transformers import AutoModelForCausalLM
from trl import DPOTrainer

sft_checkpoint = "./train/LiquidAI_LFM2_5_350M/checkpoint-1014"
dpo_checkpoint = "./train/dpo/LiquidAI_LFM2_5_350M/checkpoint-285"

policy_model = AutoModelForCausalLM.from_pretrained(
    dpo_checkpoint,
    device_map="auto",
    dtype="bfloat16",
    attn_implementation="sdpa",
)

ref_model = AutoModelForCausalLM.from_pretrained(
    sft_checkpoint,
    device_map="auto",
    dtype="bfloat16",
    attn_implementation="sdpa",
)

policy_model.config.use_cache = False
ref_model.config.use_cache = False

dpo_config.remove_unused_columns = False

test_trainer = DPOTrainer(
    model=policy_model,
    ref_model=ref_model,
    args=dpo_config,
    train_dataset=dataset_dpo["train"],
    eval_dataset=test_dataset_dpo,
    processing_class=tokenizer,
)

test_metrics = test_trainer.evaluate(metric_key_prefix="test")

print(test_metrics)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Adding EOS to eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/300 [00:00<?, ? examples/s]

Training Loss,Validation Loss,Step,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
No log,0.359267,0,0.842266,0.000000,-1.971668,-2.001568,0.710624,-0.062568,-0.952927,1.000000,0.890359,-196.756667,-354.886667


{'test_loss': 0.35926714539527893, 'eval_entropy': 0.842265625, 'eval_num_tokens': 0.0, 'eval_logits/chosen': -1.9716679775695043, 'eval_logits/rejected': -2.001567932842584, 'eval_mean_token_accuracy': 0.7106237902243933, 'eval_rewards/chosen': -0.06256789751176256, 'eval_rewards/rejected': -0.9529270650446415, 'eval_rewards/accuracies': 1.0, 'eval_rewards/margins': 0.8903591675311326, 'eval_logps/chosen': -196.75666666666666, 'eval_logps/rejected': -354.88666666666666}


In [17]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

best_dpo_checkpoint = "./train/dpo/LiquidAI_LFM2_5_350M/checkpoint-285"

tokenizer = AutoTokenizer.from_pretrained(best_dpo_checkpoint)

model = AutoModelForCausalLM.from_pretrained(
    best_dpo_checkpoint,
    device_map="auto",
    dtype=torch.bfloat16,
    attn_implementation="sdpa",
)

model.eval()
model.config.use_cache = True

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [18]:
idx = 2
example = train_dataset_dpo[idx]

messages = [
    {
        "role": "user",
        "content": example["prompt"],
    }
]

input_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    input_text,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=True,
        temperature=0.6,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
response = tokenizer.decode(generated_ids, skip_special_tokens=True)

print("PROMPT:")
print(example["prompt"])

print("\nGENERACIÓN:")
print(response)

print("\nCHOSEN:")
print(example["chosen"])

print("\nREJECTED:")
print(example["rejected"])

PROMPT:
Al amanecer,
Los cantos de los pájaros llenan el aire de melodías.
El aroma de las flores recién florecidas,
Se eleva en la suave brisa.

Los primeros rayos de luz,
Asoman entre los árboles.
El calor del sol,
Se extiende sobre la hierba con facilidad.

Al atardecer,
Los susurros del viento,
Callan el día en silencio.
El cielo se transforma en colores magníficos,
Enviando escalofríos por la espalda en la quietud.

El olor del rocío vespertino,
Se mezcla con el aire fresco.
El canto de los grillos y las ranas,
Evoca un sentido de cuidado calmante.

Los colores suaves del cielo,
Presentan un espectáculo sereno.
Una vista tan impresionante,
Los sentimientos de serenidad comienzan a fluir.

Al final del día,
La belleza de la naturaleza,
Trae alegría y paz a nuestro corazón.
Un momento precioso, para atesorar,
Y recordar cuando el mundo se desmorona. 


GENERACIÓN:
Vale la pena detenerse un segundo y escuchar ese susurro tranquilo que hace temblar la tierra Al amanecer, los cantos de

In [20]:
import os
from dotenv import load_dotenv
from huggingface_hub import login, HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

login(token=HF_TOKEN)

# Repo destino
hub_model_id = "thinkPy/dpo_LFM2_5_350M"

# Usa tu mejor checkpoint DPO
best_checkpoint_path = "./train/dpo/LiquidAI_LFM2_5_350M/checkpoint-285"
# Si tu checkpoint real tiene otro número, cámbialo aquí.

# Cargar modelo y tokenizer desde el checkpoint
model = AutoModelForCausalLM.from_pretrained(
    best_checkpoint_path,
    device_map="auto",
    dtype="bfloat16",
)

tokenizer = AutoTokenizer.from_pretrained(best_checkpoint_path)

# Recomendado para inferencia
model.config.use_cache = True

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [21]:
import os

model.push_to_hub(
    hub_model_id,
    token=HF_TOKEN,
    private=False,
)

tokenizer.push_to_hub(
    hub_model_id,
    token=HF_TOKEN,
    private=False,
)

print(f"✅ Modelo subido a: https://huggingface.co/{hub_model_id}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

✅ Modelo subido a: https://huggingface.co/thinkPy/dpo_LFM2_5_350M
